# VisionCaption
## UI 중심 시각 질의를 위한 Prompt-Fidelity Image Retrieval

- GitHub: https://github.com/soobincho-gif/VisionCaption_5
- Streamlit: https://visioncaption5.streamlit.app
- 제출 패키지 경로: `submission_package/실습5_조수빈.ipynb`

이 노트북은 VisionCaption 프로젝트의 구현 흐름, 설계 이유, 실제 실행 근거, 결과 예시를 한 번에 읽을 수 있도록 정리한 제출용 문서다. 코드 자체를 전부 다시 풀어쓰기보다, 어떤 문제를 해결하려 했고 어떤 구조로 구현했으며 어떤 결과가 나왔는지를 단계적으로 이해할 수 있도록 구성했다.

## 1. 프로젝트 개요

VisionCaption은 **자유 질의 텍스트를 입력받아 가장 적절한 이미지를 검색하고**, 그 결과가 왜 선택되었는지까지 설명하는 Streamlit 기반 포트폴리오 앱이다.

입력은 크게 두 가지다.

1. 기본 경로: 사용자가 질의를 입력하거나 benchmark quick-fill 예시를 고른다.
2. 선택 경로: 사용자가 이미지를 세션 단위로 업로드해 임시 representation과 retrieval을 실행한다.

출력은 단순한 top-1 이미지만이 아니다. 이 프로젝트는 다음을 함께 보여준다.

- top-1 결과와 top-3 결과 카드
- baseline / candidate / final default 비교
- rerank feature 기여도
- benchmark slice별 성능 지표
- query-level before/after evidence
- 업로드 이미지에 대한 session-only inspection 상태

## 2. 문제 정의와 설계 동기

일반적인 caption 기반 이미지 검색은 자연 사진에는 비교적 강하지만, **UI 스크린샷이나 역사적 대화 인터페이스처럼 텍스트·구성·엔티티가 함께 중요해지는 장면**에서는 쉽게 헷갈린다.

이 프로젝트가 해결하려는 핵심 문제는 다음과 같다.

- 단순 caption similarity만으로는 비슷한 UI 패밀리 간 오검색이 자주 발생한다.
- 같은 인물(Einstein, Cleopatra) 계열 화면에서는 이름만 맞고 실제 질의 의도는 놓칠 수 있다.
- 데모를 보여줄 때 결과만 좋다고 주장하면 설득력이 약하다. 왜 맞았는지, 어떤 근거로 바뀌었는지가 드러나야 한다.

그래서 VisionCaption은 **frozen artifact replay + deterministic rerank + evidence UI** 조합을 택했다. 이 구조 덕분에 결과 재현성이 높고, 모델 호출 없이도 benchmark 기반 설명 가능한 시연이 가능하다.

## 3. 전체 구조

아래 다이어그램은 실제 앱이 어떤 흐름으로 동작하는지 요약한 것이다.

![VisionCaption architecture](assets/diagrams/visioncaption_architecture.svg)

핵심 포인트는 다음과 같다.

- 기본 데모 경로는 frozen artifact를 사용하므로 매 실행마다 결과가 흔들리지 않는다.
- 질의는 embedding similarity로 먼저 candidate를 만들고, 필요할 때만 deterministic top-3 rerank를 적용한다.
- 업로드 이미지는 session-only 경로로 처리되며, API key가 없으면 graceful warning 상태를 보여준다.
- 결과 UI는 단순 top-1 이미지가 아니라 feature contribution, before/after trace, raw trace까지 보여준다.

In [1]:
from pathlib import Path
import json

promotion = json.loads(Path("outputs/eval/hard_best_caption_plus_selected_structured_top3_qpo025_v1/mixed_sanity_v1/default_promotion_summary.json").read_text())
report = json.loads(Path("outputs/eval/hard_best_caption_plus_selected_structured_top3_qpo025_v1/mixed_sanity_v1/report.json").read_text())
mixed_results = json.loads(Path("outputs/eval/hard_best_caption_plus_selected_structured_top3_qpo025_v1/mixed_sanity_v1/candidate_rerank_with_paraphrase/results.json").read_text())

queries = {row["query_id"]: row for row in mixed_results["queries"]}
dashboard = queries["mq001_dashboard_dark_panels"]
riverside = queries["mq034_riverside_hands_pockets"]
cake = queries["mq038_cake_miammiam"]

snapshot = {
    "promotion_status": promotion["promotion_status"],
    "default_configuration": promotion["default_configuration"],
    "mixed_sanity_result": promotion["mixed_sanity_result"],
    "hard_benchmark_result": promotion["hard_benchmark_result"],
    "query_count": report["query_count"],
    "slice_counts": report["slice_counts"],
    "cases_helped_by_paraphrase_overlap": report["cases_helped_by_paraphrase_overlap"],
}
snapshot

{'cases_helped_by_paraphrase_overlap': [{'after_top_image_id': 'history_chat_einstein_philosophical',
                                         'before_top_image_id': 'history_chat_einstein_ai',
                                         'expected_image_ids': ['history_chat_einstein_philosophical'],
                                         'query': 'beige editorial AI chat about imagination and '
                                                  'discovery rather than AI today',
                                         'query_id': 'mq010_einstein_discovery_paraphrase',
                                         'slice': 'hard_ui'}],
 'default_configuration': {'question_paraphrase_overlap': 0.25,
                           'representation_mode': 'caption_plus_selected_structured',
                           'rerank': 'deterministic_top3_rerank',
                           'rerank_top_n': 3,
                           'singleton_low_signal_container_guard_enabled': True},
 'hard_benchmark_result': {'recall_at_1': 1.0, 'recall_at_3': 1.0, 'regression_count': 0},
 'mixed_sanity_result': {'execution_mode': 'frozen_artifact_replay_based',
                         'recall_at_1': 1.0,
                         'recall_at_3': 1.0,
                         'regression_count': 0},
 'promotion_status': 'broader_default',
 'query_count': 40,
 'slice_counts': {'hard_ui': 16, 'normal_ui': 12, 'photo_or_illustration': 12}}

## 4. 전반적인 플로우

### 4-1. 기본 frozen replay 경로

1. 사용자가 질의를 입력한다.
2. 앱은 입력 문장을 그대로 provider에 보내지 않고, `resolve_replay_query()`로 가장 가까운 frozen benchmark query에 매핑한다.
3. 선택된 config(`baseline`, `candidate`, `final default`)에 따라 미리 저장된 결과 artifact를 불러온다.
4. top-1 hero, top-3 result cards, feature contribution, before/after trace를 렌더링한다.

### 4-2. 업로드 이미지 경로

1. 사용자가 이미지를 업로드하면 파일 크기와 형식을 먼저 검사한다.
2. `OPENAI_API_KEY`가 있는 경우에만 임시 caption과 embedding을 만든다.
3. 이 결과는 디스크에 저장하지 않고 session state 안에서만 유지한다.
4. 필요하면 업로드 이미지를 기존 gallery와 함께 임시 retrieval pool에 넣어 비교한다.

### 4-3. 결과 확인 경로

사용자는 단순히 "맞았다 / 틀렸다"를 보는 대신 아래까지 확인할 수 있다.

- 어떤 query가 어떤 benchmark row로 매핑되었는지
- before top-1과 after top-1이 어떻게 달라졌는지
- 어떤 feature가 rerank를 실제로 밀어 올렸는지
- 현재 선택한 config가 전체 benchmark에서 어느 정도 성능을 보이는지

## 5. 구현 로직과 핵심 설계 결정

### 5-1. 왜 frozen artifact replay를 기본 경로로 두었는가

이 프로젝트는 실시간 model call보다 **재현 가능성**을 우선했다. 제출용 앱에서 매번 embedding을 다시 만들면 결과가 흔들릴 수 있고, API 상태에 따라 시연이 깨질 수 있다. 그래서 기본 경로는 이미 검증된 artifact를 읽어와 결과를 재생하는 방식으로 설계했다.

### 5-2. 왜 representation을 caption-only에서 확장했는가

caption만으로는 UI 내부의 label-value phrase, visible text, component cue를 충분히 반영하지 못한다. 그래서 `caption_plus_selected_structured` 모드를 추가해 structured retrieval text를 embedding 입력으로 사용했다.

### 5-3. 왜 rerank를 deterministic하게 만들었는가

top-k 후보 안에 정답이 이미 들어와 있는데 top-1 순서만 어긋나는 경우가 많았다. 이때 black-box 모델을 한 번 더 호출하는 대신, exact text / entity / label-value / component / paraphrase feature를 **명시적인 가중치**로 합산하면 어떤 이유로 순서가 바뀌었는지 설명 가능해진다.

### 5-4. 상태 처리와 오류 처리

- upload 이미지는 `st.session_state` 안에서만 유지된다.
- API key가 없으면 representation 또는 embedding 단계에서 명확한 경고 메시지를 노출한다.
- artifact가 없으면 `ArtifactLoadError`로 즉시 실패시켜, 조용한 fallback 대신 문제를 드러내도록 했다.
- free-text query는 그대로 무제한 탐색하지 않고 nearest frozen query로 연결해, interactive feel과 reproducibility를 동시에 확보했다.

In [2]:
def build_embedding_source_text(
    record: CaptionRecord,
    representation_mode: CaptionRepresentationMode = "caption_only",
) -> str:
    """Return the text representation to embed for the requested mode."""
    normalized_mode = normalize_representation_mode(representation_mode)
    caption_text = _normalize_text(record.caption_text)
    structured_all_fields_text = _normalize_text(record.retrieval_text) or build_structured_retrieval_text(record)
    structured_selected_fields_text = build_selected_structured_retrieval_text(record)

    if normalized_mode == "caption_only":
        if not caption_text:
            raise ValueError(f"Caption record {record.image_id} does not contain caption_text.")
        return caption_text

    if normalized_mode == "structured_all_fields":
        if not structured_all_fields_text:
            raise ValueError(f"Caption record {record.image_id} does not contain structured retrieval_text.")
        return structured_all_fields_text

    if normalized_mode == "structured_selected_fields":
        if not structured_selected_fields_text:
            raise ValueError(f"Caption record {record.image_id} does not contain selected structured retrieval text.")
        return structured_selected_fields_text

    if normalized_mode == "caption_plus_selected_structured":
        candidate_text = build_candidate_baseline_retrieval_text(record)

위 코드는 embedding에 실제로 어떤 문자열을 넣을지를 결정하는 부분이다. `caption_only`, `structured_all_fields`, `structured_selected_fields`, `caption_plus_selected_structured`를 분기하며, 최종 default에서는 `caption_plus_selected_structured`를 사용한다. 즉, 이 프로젝트의 핵심은 모델을 더 크게 바꾸는 것이 아니라 **입력 표현을 retrieval 목적에 맞게 바꾼 것**에 가깝다.

In [3]:
class DeterministicRerankWeights:
    """Explicit rerank weights kept small and easy to edit."""

    exact_visible_text_overlap: float = 1.2
    named_entity_match: float = 1.5
    label_value_phrase_overlap: float = 2.0
    component_cue_overlap: float = 1.3
    question_paraphrase_overlap: float = 0.0
    original_similarity: float = 0.25

    def as_dict(self) -> dict[str, float]:
        """Return a JSON-friendly view of the configured weights."""
        return asdict(self)


DEFAULT_DETERMINISTIC_RERANK_WEIGHTS = DeterministicRerankWeights()


def _normalize_text(value: str | None) -> str:
    """Collapse text into a stable lowercase form for exact matching."""
    return " ".join(str(value or "").lower().split())


def _tokenize(value: str) -> list[str]:
    """Tokenize user-facing text while removing generic glue words."""
    return [token for token in WORD_PATTERN.findall(_normalize_text(value)) if token not in QUERY_STOPWORDS]


def _tokenize_question_paraphrase(value: str) -> list[str]:
    """Tokenize question-like cues while dropping generic interface/topic words."""
    return [
        token
        for token in _tokenize(value)
        if token not in QUESTION_PARAPHRASE_LOW_SIGNAL_TOKENS
    ]


def _split_visible_text_item(value: str) -> list[str]:
    """Recover short cues from overlong OCR-like visible-text strings."""
    fragments = [_normalize_text(value)]
    for pattern in (VISIBLE_TEXT_SECTION_PATTERN, VISIBLE_TEXT_SEPARATOR_PATTERN):
        next_fragments: list[str] = []
        for fragment in fragments:
            pieces = [
                _normalize_text(piece.strip(" -|"))
                for piece in pattern.split(fragment)
                if _normalize_text(piece.strip(" -|"))
            ]
            next_fragments.extend(pieces if len(pieces) > 1 else [fragment])
        fragments = next_fragments

    deduped_fragments: list[str] = []
    for fragment in fragments:
        if len(fragment) < 4 or fragment in deduped_fragments:
            continue
        deduped_fragments.append(fragment)

    return deduped_fragments


def _looks_like_interface(record: CaptionRecord) -> bool:
    """Restrict UI-style features to interface-like records."""
    image_type = _normalize_text(record.image_type)
    if any(keyword in image_type for keyword in INTERFACE_IMAGE_TYPE_KEYWORDS):
        return True
    return bool(record.layout_blocks)


def extract_exact_text_cues(record: CaptionRecord) -> list[str]:
    """Return compact visible-text cues for exact phrase overlap."""
    prioritized_fragments: list[str] = []
    regular_fragments: list[str] = []
    for visible_text_item in record.visible_text:
        fragments = _split_visible_text_item(visible_text_item)
        if len(fragments) > 1 or len(visible_text_item) > 80 or "?" in visible_text_item:
            prioritized_fragments.extend(fragments)
            continue
        regular_fragments.extend(fragments)

    exact_text_cues: list[str] = []
    for fragment in prioritized_fragments:
        if "." in fragment and "?" not in fragment:
            continue
        if "," in fragment and "?" not in fragment:
            continue
        if "?" not in fragment and len(fragment.split()) > 8:
            continue
        if fragment not in exact_text_cues:
            exact_text_cues.append(fragment)

    for fragment in regular_fragments:

이 부분은 rerank 쪽 핵심이다. 가중치가 아주 크지 않고, 각각의 feature가 무엇을 보는지 명확히 드러난다. 이 덕분에 결과가 바뀌었을 때 "왜 바뀌었는지"를 설명할 수 있고, 회귀가 생기면 어떤 feature가 과하게 작동했는지도 추적할 수 있다.

In [4]:
def load_portfolio_artifacts() -> PortfolioArtifacts:
    """Load and cache the full artifact bundle needed by the UI."""
    return PortfolioArtifacts(
        repo_root=REPO_ROOT,
        promotion_summary=read_json(PROMOTION_SUMMARY_PATH),
        mixed_validation_report=read_json(MIXED_VALIDATION_REPORT_PATH),
        captions_by_id=_load_captions_by_id(),
        benchmarks={spec.key: _load_benchmark_bundle(spec) for spec in BENCHMARK_SPECS.values()},
    )


def get_representation_text(caption_record: dict[str, Any], representation_mode: str) -> str:
    """Return the text representation used by a selected config."""
    if representation_mode == "caption_only":
        return str(caption_record.get("caption_text", "")).strip()
    return str(caption_record.get("retrieval_text") or caption_record.get("caption_text", "")).strip()


def get_config_metrics(config_artifacts: ConfigArtifacts) -> dict[str, Any]:
    """Return a normalized metrics mapping across config artifact shapes."""
    if "recall_at_1" in config_artifacts.results:
        return config_artifacts.results
    return dict(config_artifacts.results.get("after_metrics", {}))


def build_query_rows(bundle: BenchmarkBundle, config_key: str) -> list[dict[str, Any]]:
    """Merge benchmark metadata and top-k results into a table-friendly row list."""
    config_artifacts = bundle.configs[config_key]
    result_queries = {
        query["query_id"]: dict(query)
        for query in config_artifacts.results.get("queries", [])
    }
    rows: list[dict[str, Any]] = []
    for query_id in bundle.query_order:
        benchmark_query = dict(bundle.queries_by_id[query_id])
        result_query = dict(result_queries.get(query_id, {}))
        results = list(result_query.get("results", []))
        if not results:
            results = list(result_query.get("reranked_results", []))
        if not results:
            results = list(result_query.get("top_results", []))
        results = [_normalize_result_image_path(result, bundle.images_by_id) for result in results]
        expected_image_ids = benchmark_query.get("expected_image_ids", [])
        top_result = results[0] if results else None
        second_result = results[1] if len(results) > 1 else None
        top1_correct = bool(top_result and top_result.get("image_id") in expected_image_ids)
        margin = None
        if top_result and second_result:
            margin = float(top_result["similarity_score"]) - float(second_result["similarity_score"])
        rows.append(
            {
                **benchmark_query,
                **result_query,
                "query_id": query_id,
                "results": results,
                "top_result": top_result,
                "second_result": second_result,
                "top1_correct": top1_correct,
                "top1_margin": margin,
                "failure_bucket": derive_failure_bucket(
                    tags=benchmark_query.get("tags", []),
                    top1_correct=top1_correct,
                    top_result_image_id=top_result.get("image_id") if top_result else None,
                    hard_negative_image_ids=benchmark_query.get("hard_negative_image_ids", []),
                ),
                "query_log": config_artifacts.query_logs_by_id.get(query_id),
            }
        )
    return rows


def _normalize_result_image_path(
    result: dict[str, Any],
    images_by_id: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    """Prefer deploy-safe benchmark image paths over stale artifact-local paths."""
    normalized = dict(result)
    image_id = str(normalized.get("image_id") or "")
    if image_id in images_by_id:
        normalized["image_path"] = images_by_id[image_id]["image_path"]
    elif normalized.get("image_path"):
        normalized["image_path"] = portable_artifact_path(str(normalized["image_path"]))
    return normalized


def derive_failure_bucket(
    tags: list[str],
    top1_correct: bool,
    top_result_image_id: str | None,
    hard_negative_image_ids: list[str],
) -> str:
    """Assign one primary failure bucket for filtering and table views."""
    if top1_correct:
        return "Correct"
    tag_set = set(tags)
    if "person_name" in tag_set:
        return "Person-name confusion"
    if "visible_text" in tag_set:
        return "Visible-text confusion"
    if "layout_structure" in tag_set:
        return "Layout confusion"
    if top_result_image_id and top_result_image_id in set(hard_negative_image_ids):
        return "Same-family UI confusion"
    if "paraphrase" in tag_set:
        return "Paraphrase miss"
    if "illustration_style" in tag_set:
        return "Illustration mismatch"
    if "photo_scene" in tag_set:
        return "Photo scene mismatch"
    return "Other retrieval miss"


def tokenize(value: str) -> set[str]:
    """Tokenize text into lowercase alphanumeric terms."""
    return {token for token in WORD_PATTERN.findall(normalize_text(value))}


def resolve_replay_query(query_text: str, bundle: BenchmarkBundle) -> tuple[dict[str, Any], dict[str, Any]]:
    """Map free text to the closest frozen benchmark query for offline-safe replay."""
    normalized_input = normalize_text(query_text)
    if not normalized_input:
        first_query = bundle.queries_by_id[bundle.query_order[0]]
        return first_query, {"exact_match": True, "score": 1.0, "matched_query_id": first_query["query_id"]}

    input_tokens = tokenize(normalized_input)
    best_query = bundle.queries_by_id[bundle.query_order[0]]
    best_score = -1.0
    exact_match = False

    for query in bundle.queries_by_id.values():
        candidate_text = query["query"]
        normalized_candidate = normalize_text(candidate_text)
        candidate_tokens = tokenize(candidate_text)
        token_overlap = 0.0
        if input_tokens or candidate_tokens:
            token_overlap = len(input_tokens & candidate_tokens) / max(len(input_tokens | candidate_tokens), 1)
        phrase_similarity = SequenceMatcher(None, normalized_input, normalized_candidate).ratio()
        score = (token_overlap * 0.7) + (phrase_similarity * 0.3)
        if normalized_input == normalized_candidate:
            score = 2.0

UI 유틸리티 계층에서는 artifact를 한 번에 읽고, query metadata와 결과 row를 합쳐서 화면용 데이터 구조로 바꾼다. 이 레이어가 있기 때문에 페이지 컴포넌트는 파일 구조를 직접 알 필요 없이 `bundle`, `rows`, `query_log` 단위로 데이터를 사용할 수 있다.

## 6. 실행 과정과 검증 로그

실제 제출 직전에는 두 가지를 우선 확인했다.

1. 테스트가 모두 통과하는지
2. Streamlit 앱이 로컬에서 부팅되는지

아래 로그는 그 확인 과정을 요약한 것이다.

In [5]:
# verification logs
print(pytest_log)
print("
---
")
print(streamlit_log)

/Users/sarahc/.pyenv/versions/3.12.8/lib/python3.12/site-packages/pytest_asyncio/plugin.py:208: PytestDeprecationWarning: The configuration option "asyncio_default_fixture_loop_scope" is unset.
  warnings.warn(PytestDeprecationWarning(_DEFAULT_FIXTURE_LOOP_SCOPE_UNSET))
...................                                                      [100%]
19 passed in 0.06s

---

$ streamlit run app.py --server.headless true --server.port 8510
  You can now view your Streamlit app in your browser.
  Local URL: http://localhost:8510


## 7. 정량 결과

최종 default는 mixed sanity benchmark에서 다음과 같은 progression으로 향상되었다.

- `caption_only`: Recall@1 67.5%
- `caption_plus_selected_structured`: Recall@1 87.5%
- `caption_plus_selected_structured + deterministic_top3_rerank`: Recall@1 97.5%
- `caption_plus_selected_structured + deterministic_top3_rerank + question_paraphrase_overlap=0.25`: Recall@1 100.0%

즉, 이 프로젝트는 한 번에 큰 점프를 만든 것이 아니라, **표현 개선 -> rerank 추가 -> paraphrase cue 추가** 순서로 오류 원인을 줄여 가며 성능을 올린 구조다.

In [6]:
ablation_rows = report["ablation_rows"]
ablation_rows

시스템,Recall@1,Recall@3,이전 단계 대비 변화,이전 단계 대비 회귀
caption_only,67.5%,100.0%,baseline,N/A
caption_plus_selected_structured,87.5%,100.0%,20.0%,4
caption_plus_selected_structured + deterministic_top3_rerank,97.5%,100.0%,10.0%,0
caption_plus_selected_structured + deterministic_top3_rerank + question_paraphrase_overlap=0.25,100.0%,100.0%,2.5%,0


## 8. 결과 예시 1: 대시보드 계열 오검색 교정

이 사례는 비슷한 UI 패밀리끼리 헷갈리는 문제를 보여준다. 초기 candidate에서는 `visual_storytelling_mobile`이 rank 1이었지만, label-value phrase와 component cue를 본 rerank 이후 `visual_storytelling_dashboard`가 rank 1로 올라왔다.

<table><tr>
<td><img src="assets/sample_images/visual_storytelling_mobile.png" width="320"><br><b>Before top-1</b><br>visual_storytelling_mobile</td>
<td><img src="assets/sample_images/visual_storytelling_dashboard.png" width="320"><br><b>After top-1</b><br>visual_storytelling_dashboard</td>
</tr></table>

In [7]:
dashboard_case = {
    "query_id": dashboard["query_id"],
    "query": dashboard["query"],
    "activation_reasons": dashboard["activation_reasons"],
    "before_top_1": dashboard["before_top_result"]["image_id"],
    "after_top_1": dashboard["after_top_result"]["image_id"],
    "before_top_3": [row["image_id"] for row in dashboard["original_results"]],
    "after_top_3": [row["image_id"] for row in dashboard["reranked_results"]],
    "corrected": dashboard["corrected"],
}
dashboard_case

{'activation_reasons': ['label_value_phrase_overlap', 'component_cue_overlap'],
 'after_top_1': 'visual_storytelling_dashboard',
 'after_top_3': ['visual_storytelling_dashboard',
                 'visual_storytelling_mobile',
                 'history_chat_cleopatra_landing'],
 'before_top_1': 'visual_storytelling_mobile',
 'before_top_3': ['visual_storytelling_mobile',
                  'visual_storytelling_dashboard',
                  'history_chat_cleopatra_landing'],
 'corrected': True,
 'query': 'dark dashboard for visual storytelling from images and sentiment with file list, '
          'controls, image descriptions, and pipeline status',
 'query_id': 'mq001_dashboard_dark_panels',
 'regression': False}

## 9. 결과 예시 2: 야간 인물 사진 사례

이 사례는 UI가 아닌 일반 사진 장면에서도 retrieval이 안정적으로 동작한다는 점을 보여준다. 질의의 핵심 단서는 `야간`, `강변`, `흐릿한 인물`, `도시 불빛` 같은 장면 정보이며, final default에서도 top-1이 흔들리지 않는다. 즉, 이 프로젝트는 복잡한 UI 장면만 잘 맞추는 것이 아니라, 사진 장면에 대해서도 representation과 similarity만으로 안정적인 검색을 수행할 수 있다.

<img src="assets/sample_images/night_riverside_person.jpeg" width="320">


In [8]:
riverside_case = {
    "query_id": riverside["query_id"],
    "query": riverside["query"],
    "activation_reasons": riverside["activation_reasons"],
    "before_top_1": riverside["before_top_result"]["image_id"],
    "after_top_1": riverside["after_top_result"]["image_id"],
    "before_top_3": [row["image_id"] for row in riverside["original_results"]],
    "after_top_3": [row["image_id"] for row in riverside["reranked_results"]],
    "corrected": riverside["corrected"],
}
riverside_case

{'activation_reasons': [],
 'after_top_1': 'night_riverside_person',
 'after_top_3': ['night_riverside_person', 'outdoor_blanket_rest', 'cafe_phone_person'],
 'before_top_1': 'night_riverside_person',
 'before_top_3': ['night_riverside_person', 'outdoor_blanket_rest', 'cafe_phone_person'],
 'corrected': False,
 'query': 'blurred person in a dark jacket with hands in pockets on a riverside slope after sunset',
 'query_id': 'mq034_riverside_hands_pockets',
 'regression': False}

## 10. 결과 예시 3: 텍스트가 포함된 실사진 사례

케이크 사례는 일반 사진이지만, 동시에 텍스트 단서가 포함된 이미지라는 점에서 흥미롭다. 이 프로젝트는 UI만 다루는 것이 아니라, 사진 장면 안에 들어 있는 텍스트와 시각 단서를 함께 반영해 retrieval을 수행한다. 아래 예시는 `miammiam` 표기와 손글씨 메시지가 함께 보이는 케이크 이미지를 안정적으로 유지한 사례다.

<img src="assets/sample_images/cake_korean_message.jpeg" width="320">

추가 샘플 갤러리

<table><tr>
<td><img src="assets/sample_images/cafe_phone_person.jpeg" width="220"><br><b>Cafe scene</b></td>
<td><img src="assets/sample_images/indoor_many_cats.jpeg" width="220"><br><b>Indoor cats</b></td>
<td><img src="assets/sample_images/outdoor_blanket_rest.jpeg" width="220"><br><b>Outdoor rest</b></td>
<td><img src="assets/sample_images/illustrated_couple_sofa.jpeg" width="220"><br><b>Illustration scene</b></td>
</tr></table>


In [9]:
cake_case = {
    "query_id": cake["query_id"],
    "query": cake["query"],
    "activation_reasons": cake["activation_reasons"],
    "before_top_1": cake["before_top_result"]["image_id"],
    "after_top_1": cake["after_top_result"]["image_id"],
    "before_top_3": [row["image_id"] for row in cake["original_results"]],
    "after_top_3": [row["image_id"] for row in cake["reranked_results"]],
    "corrected": cake["corrected"],
    "regression": cake["regression"],
}
cake_case

{'activation_reasons': [],
 'after_top_1': 'cake_korean_message',
 'after_top_3': ['cake_korean_message', 'cafe_phone_person', 'outdoor_blanket_rest'],
 'before_top_1': 'cake_korean_message',
 'before_top_3': ['cake_korean_message', 'cafe_phone_person', 'outdoor_blanket_rest'],
 'corrected': False,
 'query': 'birthday cake with miammiam heart label and handwritten Korean message on the board',
 'query_id': 'mq038_cake_miammiam',
 'regression': False}

## 11. 실제 앱 화면 예시

아래 스크린샷은 제출 시점에 로컬에서 캡처한 실제 실행 화면이다. Overview, Live Demo, upload 영역, Benchmark Explorer, Method & Architecture까지 전체 흐름이 이어진다.

### 11-1. Overview
![overview](assets/screenshots/01_overview.png)

### 11-2. Live Demo 기본 실행 상태
![live-demo](assets/screenshots/02_live_demo_results.png)

### 11-3. Session-only upload 영역
![upload-state](assets/screenshots/03_live_demo_upload_warning.png)

### 11-4. Benchmark Explorer
![benchmark-explorer](assets/screenshots/04_benchmark_explorer.png)

### 11-5. Method & Architecture
![method-architecture](assets/screenshots/05_method_architecture.png)

## 12. 구현 진행 흐름

이 프로젝트는 한 번에 완성된 것이 아니라, benchmark와 artifact를 기준으로 작은 단계를 쌓아가며 발전했다. 문서 로그 기준으로 보면 다음 흐름으로 정리할 수 있다.

In [10]:
for step in milestones:
    print(step)

2026-04-09_01_repository_scaffold.md -> 저장소 기본 구조와 앱 진입점 정리
2026-04-09_09_prompt_fidelity_benchmark.md -> 문제를 재현할 수 있는 benchmark 정의
2026-04-09_10_structured_representation_experiment.md -> 구조화 표현 실험 추가
2026-04-10_04_top3_rerank_ablation.md -> deterministic top-3 rerank ablation 수행
2026-04-10_05_mixed_sanity_validation.md -> broader mixed sanity validation 수행
2026-04-10_06_default_promotion_offline_repro.md -> frozen replay 기반 default 확정


## 13. 한계와 개선점

### 현재 한계

- 기본 경로는 frozen replay 기반이라, 실시간 사용자 자유 질의를 무한정 탐색하는 일반 검색 엔진과는 성격이 다르다.
- 업로드 경로는 API key가 없으면 representation과 embedding을 만들 수 없다.
- deterministic rerank는 현재 benchmark failure pattern에 맞춰 설계되었기 때문에, 더 넓은 도메인으로 확장할 때는 추가 검증이 필요하다.

### 다음 개선 방향

- 더 큰 benchmark pool과 더 다양한 hard negative를 추가해 일반화 성능 확인
- upload path의 실사용성 개선과 richer failure messaging 정리
- benchmark explorer의 query evidence를 더 안정적인 card/grid 컴포넌트로 세분화
- Streamlit deployment에서 anonymous public access 상태를 지속적으로 유지할 수 있도록 공유 설정 점검

## 14. 마무리 정리

VisionCaption의 핵심은 단순히 이미지를 검색하는 것이 아니라, **왜 그 결과가 선택되었는지까지 설명 가능한 retrieval UI를 만드는 것**이었다.

정리하면 이 프로젝트는 다음을 달성했다.

- UI 중심 시각 질의를 다루는 benchmark와 artifact pipeline 구축
- structured representation 기반 candidate 품질 향상
- deterministic top-3 rerank로 top-1 ordering 문제 해결
- paraphrase cue를 포함한 broader default 검증 완료
- Streamlit UI, README, 제출용 notebook까지 포함한 제출 패키지 정리

즉, 이 프로젝트는 모델 성능 개선과 동시에 **재현성, 설명 가능성, 시연 가능성**을 함께 갖춘 retrieval system으로 마무리되었다.